In [1]:
import copy
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os
import draw.basic_functio.motif as motif
from draw.basic_functio.get_rectangular_size_interval import calc_envelope_for_group
import  copy
os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())
from config import DATA_DIR,INPUT_DIR
from pathlib import Path


backend (before pyplot): module://matplotlib_inline.backend_inline
backend (after pyplot): QtAgg


In [2]:
from typing import Dict, Any

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [basicSa, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }





In [3]:
SIMULATION_EDITION = 'motif2'

In [4]:


time_2_build = 60
TIME_2_BUILD = time_2_build

# 默认："{SIMULATION_EDITION}/topology_{TIME_2_BUILD}"；可用环境变量 TOPOLOGY_VERSION 覆盖
DEFAULT_VERSION = f"{SIMULATION_EDITION}/topology_{TIME_2_BUILD}"
VERSION = os.getenv("TOPOLOGY_VERSION", DEFAULT_VERSION)

RAW_DIR    = Path(INPUT_DIR) / VERSION / "raw"
MODIFY_DIR    = Path(INPUT_DIR) / VERSION / "modify"

CONFIG_DIR = Path(INPUT_DIR) / f"{SIMULATION_EDITION}/config"
MODIFY_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# CONFIG_DIR

WindowsPath('C:/usrspace/mywork/generic/data/input/motif2/config')

In [5]:

# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml  as read_snap_xml
import draw.read_snap_xml  as read_snap_xml
# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)

In [6]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"



In [7]:
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

# ====================== 读取数据 ======================

# start_ts = 10717
# # end_ts   = 86399
# end_ts   = 11640
# # 解析 XML 得到 group_data，结构：{time_step: {'groups': {...}}}
# group_data = read_snap_xml.parse_xml_group_data(xml_file, start_ts, end_ts)
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 30152
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)

#下面是图变换的。
#

In [8]:
# base_groupid_now = 1

In [9]:
# rev_group_data,offset = read_snap_xml.modify_group_data(group_data,P=18, N=36, base_groupid=base_groupid_now)

NameError: name 'group_data' is not defined

下面主要是为了测试检测我们的图

In [8]:

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []

In [17]:


# # 3) 创建并配置 viewer
# viewer = SatelliteViewer(group_data)
# viewer.setWindowTitle("group_data")
# viewer.resize(1200, 700)
# # viewer.edges_by_step =
# # viewer.pending_links_by_step =
# viewer.show()



In [24]:
# # 3) 创建并配置 viewer
# viewer = SatelliteViewer(rev_group_data)
# viewer.setWindowTitle("standard rev_group_data")
# viewer.resize(1200, 700)
# # viewer.edges_by_step =
# # viewer.pending_links_by_step =
# viewer.show()


In [25]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

In [26]:

# get the rectangular size of the group
# # 同样是计算block尺寸
# import draw.basic_functio.get_rectangular_size_interval as get_rectangular_size_interval
# t1,t2=get_rectangular_size_interval.calc_envelope_for_group(rev_group_data,[start_ts,end_ts],0,P,N)
# t3,t4=get_rectangular_size_interval.calc_envelope_for_group(rev_group_data,[start_ts,end_ts],4,P,N)
#
#
# # ====================== 静态包络矩形 ======================
# #可选
# # rects = {
# #     4: [(9, 15, 32, 35)],
# #     0: [(0, 5, 9, 13), (17, 17, 30, 32)],
# # }
# rects = {
#     0: (t1, t2),
#     4: (t3, t4),
# }
#
# colors = {4: "deeppink", 0: "orange"}  # 可选
#
# viewer.show_envelopes_static(
#     rects_by_group=rects,
#     expand=0.35,
#     colors=colors,
#     persist=True
# )

下面主要是为了确认，我们的包络矩形是否正确

## 微调时间区间
早些时候的时间区间是通过包络大大小去确定的，但是我们发现，包络矩形的位置也在变化，有时候单纯以大的包络矩形来覆盖位置变化不够精确，因此，在这一步，我们往往会根据包络矩形位置以及大小进一步的细分调整时间.
注意，这里，先大致上判断要细调哪个区间


In [ ]:
# df = get_rectangular_size_interval.get_envelope_traces(rev_group_data, groupid=4, P=18, N=36, t0=start_ts, t1=end_ts)
#
# # 2. 可直接保存df到csv
# # df.to_csv('envelope_trace_group4.csv', index=False)
#
# # 3. 画随时间变化的包络范围
# get_rectangular_size_interval.plot_envelope_traces(df)

In [ ]:

# ====================== 计算 Block 尺寸 ======================
import draw.basic_show.get_satellite_block_info as get_satellite_block_info


# times = sorted(group_data.keys())   # 时间序列（所有 step）
#
# # 下述代码是为了计算某个区域的块分布
# groupid = 0  # 目标分组 ID
#
# # 存储 Block1/Block2 的长宽随时间变化
# widths1, heights1, widths2, heights2 = [], [], [], []
# for t in times:
#     sats4 = group_data[t]['groups'][groupid]
#     # 返回：Block1 (长度, 宽度)，Block2 (长度, 宽度)
#     (l1, w1), (l2, w2) = get_satellite_block_info.get_satellite_block_info(sats4, P, N)
#     widths1.append(l1); heights1.append(w1)
#     widths2.append(l2); heights2.append(w2)
#
# # ====================== 可视化 ======================
# # 创建上下两个子图：Block1 和 Block2
# fig, (ax1, ax2) = plt.subplots(
#     2, 1, figsize=(12, 8), sharex=True,
#     gridspec_kw={'height_ratios': [1, 1]}
# )
# # -------- Block1 曲线 --------
# ax1.plot(times, widths1,  label='Block1 Length', color='tab:blue')
# ax1.plot(times, heights1, label='Block1 Width',  color='tab:orange')
# ax1.set_ylabel('Block1 Size')
# ax1.set_title('Block1（一般为主包络）长宽随时间变化')
# ax1.legend()
# ax1.grid(alpha=0.3)
# # -------- Block2 线 --------
# ax2.plot(times, widths2,  label='Block2 Length', color='tab:green')
# ax2.plot(times, heights2, label='Block2 Width',  color='tab:red')
# ax2.set_ylabel('Block2 Size')
# ax2.set_title('Block2（有时不存在）长宽随时间变化')
# ax2.legend()
# ax2.grid(alpha=0.3)
# # 公共横轴
# plt.xlabel('Time Step')
# # 自动调整布局，避免文字重叠
# plt.tight_layout()
# # 显示图形（独立 Qt 窗口，不阻塞控制台）
# plt.show()


## 同构图规划
确定好包络矩形后，我们就可以开始规划同构图拓扑了

下面两段代码，分别是研发和对比试验，通过这样的操作，我们能顾极大的减少我们研究的步骤


下面是同构图验证，只需要查看一个拓扑图即可，

这里有一个额外的代码，那就是温和过渡。
所谓的温和过渡，就是在某些情况下，按照上述老方法，会出现，一口气切换一大块链路，但实际上，没必要一口气切换这么大的链路，因为会造成网络的不稳定性。

下述的代码，一般情况下是不需要运行的，一般是到后面进行微调才运行的


In [51]:
import draw.basic_functio.topology_config as topology_config

rec = topology_config.TopologyRecorder(P, N)
rec.modify_group_data(group_data, base_groupid=base_groupid_now)

nodes = {}

# 永久（无时间窗）规则：保持原写法
# rec.write_distinct_motif(0, 17, 0, 8, nodes, option=1)
# rec.write_distinct_motif(0, 17, 9, 29, nodes, option=0)
# rec.write_distinct_motif(0,  9, 30, 31, nodes, option=0)
# rec.write_distinct_motif(15,17, 32, 35, nodes, option=2)
# rec.write_distinct_motif(0, 17, 32, 35, nodes, option=0)
#
# # 动态（有时间窗）规则：t_start/t_end 可以写表达式字符串
# rec.write_distinct_motif_with_time(9, 17, 30, 31, nodes, t_start=1204, t_end=1232, option=2)
#



rec.write_distinct_motif(0, 17, 0, 8, nodes, option=1)
rec.write_distinct_motif(0, 17, 9, 35, nodes, option=0)




# 保存时“symbols”可选（只是提示配置里用到了哪些符号）
rec.save(CONFIG_DIR / f"{start_ts}_{end_ts}.json",
         symbols=["start_ts", "end_ts", "time_2_build"])
env = {
    "start_ts": start_ts,
    "end_ts": end_ts,
    "time_2_build": time_2_build,
}
all_rev_inter_edge = rec.render_adj_range(start_ts, end_ts, eval_env=env)

In [29]:
viewer = SatelliteViewer(rev_group_data)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = all_rev_inter_edge

viewer.show()

# get the rectangular size of the group
# 同样是计算block尺寸
import draw.basic_functio.get_rectangular_size_interval as get_rectangular_size_interval
t1,t2=get_rectangular_size_interval.calc_envelope_for_group(rev_group_data,[start_ts,end_ts],0,P,N)
t3,t4=get_rectangular_size_interval.calc_envelope_for_group(rev_group_data,[start_ts,end_ts],4,P,N)


# ====================== 静态包络矩形 ======================
#可选
# rects = {
#     4: [(9, 15, 32, 35)],
#     0: [(0, 5, 9, 13), (17, 17, 30, 32)],
# }
rects = {
    0: (t1, t2),
    4: (t3, t4),
}

colors = {4: "deeppink", 0: "orange"}  # 可选

viewer.show_envelopes_static(
    rects_by_group=rects,
    expand=0.35,
    colors=colors,
    persist=True
)


In [34]:
end_ts

3669

In [31]:
import draw.basic_functio.topology_config as topology_config

cfg = topology_config.load_config(CONFIG_DIR / f"{start_ts}_{end_ts}.json")

# 渲染时提供变量环境（可以动态变更 time_2_build 等）
env = {
    "start_ts": start_ts,
    "end_ts": end_ts,
    "time_2_build": time_2_build,
}

rec = topology_config.TopologyRecorder(cfg.P, cfg.N)
rec.base_groupid = cfg.base_groupid
rec._motifs = cfg.motifs

# 渲染某一秒的邻接
# adj_1232 = rec.render_adj_at(1232, eval_env=env)

# 渲染整段并生成 all_rev_inter_edge（你的老变量名）
all_rev_inter_edge = rec.render_adj_range(start_ts, end_ts, eval_env=env)


In [32]:
viewer = SatelliteViewer(rev_group_data)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = all_rev_inter_edge

viewer.show()
_viewer_list.append(viewer)
viewer.show_envelopes_static(
    rects_by_group=rects,
    expand=0.35,
    colors=colors,
    persist=True
)

### 处理冲突链路
上述获得的是原始图的边，但是，其存在冲突问题，例如 t=100s是a与b t=101s就是a与c了，考虑到链路建链需要时间，因此，我们这里需要进行一些冲突修正
假设建链时间是setuptime  =60s
1. 若a与b是区域内链路，a与c是是区域外链路，由于区域内链路是必须保留的，因此，我们认为，a与c的链路是从t=101才开始建链，并且t=160末尾才完成建链，并在t=161时投入使用
2. 若a与b是区域外链路，a与c是区域内链路，由于区域内链路是必须保留的，因此，我们认为，a与c的链路在t=101时刻就可投入使用，因此，a与b的链路已经在前60s断链，认为a与c的链路在t=101-60也就是y=41s时开始建链



In [33]:

# 注意上述我们是在同构拓扑序列上进行的，因此，我们还要将同构拓扑序列进行还原，同时，我们还要考虑到建链时间约束
import draw.basic_functio.revdata2rawdata as revdata2rawdata
# attention ,here  it just composed of the inter-link, intra_link hasn't benn conclued
raw_inter_edge = revdata2rawdata.revedge2rawedge(all_rev_inter_edge,offset)

In [34]:


import draw.basic_functio.conflict_link as conflict_link
# # 这个是目前最新的，仍然有小bug，但是暂时不修了，等后面再修
raw_edges_by_step,pending_edges = conflict_link.get_no_conflict_link(raw_inter_edge,offset,rects,start_ts,end_ts,time_2_build,N,P)


In [35]:
viewer = SatelliteViewer( group_data)
viewer.setWindowTitle("group_data   ")
viewer.resize(1200, 700)
viewer.edges_by_step = raw_edges_by_step
viewer.pending_links_by_step =pending_edges
viewer.show()

我们要处理好建链时间冲突，因此，下面就是处理冲突的代码

In [78]:

# 下面是把边转为node存储，因为这种方式存储会比较方便


import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
all_nodes = inter_edge2nodes.trans_edge2node_test(raw_edges_by_step,P,N)

In [69]:
import draw.basic_functio.write2xml as write2xml
import genaric2.tegnode as tegnode
from copy import deepcopy

start1 = 18396

end1 = 19831


file_path = RAW_DIR / f"interplane_links_{start1}_{end1}.xml"


nodes1 = write2xml.xml_to_nodes_test(file_path, tegnode.tegnode_new)

start2 =19831

end2 =20814


file_path = RAW_DIR / f"interplane_links_{start2}_{end2}.xml"


nodes_middle = write2xml.xml_to_nodes_test(file_path, tegnode.tegnode_new)



start3 =20814

end3 =22005

file_path = RAW_DIR / f"interplane_links_{start3}_{end3}.xml"
nodes3 = write2xml.xml_to_nodes_test(file_path, tegnode.tegnode_new)
#


In [70]:
start_ts_onestep = start1
# end_ts   = 86399Q
end_ts_onestep   = end3

group_data_one_step = slice_group_data(raw_group_data, start_ts_onestep, end_ts_onestep)

In [71]:




nodes2 =nodes_middle
totalnode_raw = {**nodes1, **nodes2,**nodes3 }



In [52]:
from typing import Dict, Iterable, Set, Tuple

def _edges_from_snapshot(snapshot: Dict[int, Iterable[int]], *, undirected: bool=True) -> Set[Tuple[int, int]]:
    """
    snapshot 形如 {u: {v1, v2, ...}, ...}
    返回该快照的边集合：
      - 无向：用排序后的 (min(u,v), max(u,v)) 去重
      - 有向：用 (u, v)
    """
    E: Set[Tuple[int, int]] = set()
    for u, nbrs in snapshot.it


viewer.show()
_viewer_list.append(viewer)

# totalnode_comple = copy.deepcopy(totalnode_raw)

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
# 注意 ，下面是直接将nodes转为edge，因为我们的nodes本身已经完成了冲突检测和处理
# import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
# three_raw_all_inter_edge = inter_edge2nodes.trans_nodes2edges(totalnode_comple,P,N)
boundary_right = 3669
boundary_left = 3668

three_raw_all_inter_edge[3669]
from typing import Dict, Iterable, Set, Tuple


def _edges_from_snapshot(snapshot: Dict[int, Iterable[int]], *, undirected: bool = True) -> Set[Tuple[int, int]]:
    """
    snapshot 形如 {u: {v1, v2, ...}, ...}
    返回该快照的边集合：
      - 无向：用排序后的 (min(u,v), max(u,v)) 去重
      - 有向：用 (u, v)
    """
    E: Set[Tuple[int, int]] = set()
    for u, nbrs in snapshot.items():
        if nbrs is None:
            continue
        for v in nbrs:
            if u == v:
                continue
            if undirected:
                a, b = (u, v) if u < v else (v, u)
                E.add((a, b))
            else:
                E.add((u, v))
    return E

def diff_links(
    data: Dict[int, Dict[int, Iterable[int]]],
    left: int,
    right: int,
    *,
    undirected: bool=True
) -> Dict[str, list]:
    """
    比较 data[right] 相对 data[left] 的链路变化。
    返回:
      - added:  right 有而 left 没有（新增）
      - removed: left 有而 right 没有（消失）
      - unchanged: 两边都存在（保持）
    """
    snapL = data.get(left, {})   # 3668
    snapR = data.get(right, {})  # 3669
    EL = _edges_from_snapshot(snapL, undirected=undirected)
    ER = _edges_from_snapshot(snapR, undirected=undirected)

    added = sorted(ER - EL)
    removed = sorted(EL - ER)
    unchanged = sorted(ER & EL)
    return {"added": added, "removed": removed, "unchanged": unchanged}


In [22]:
# # 这个是目前最新的，仍然有小bug，但是暂时不修了，等后面再修
# raw_edges_by_step_onestep,pending_edges_onstep,modifynodes_onestep = conflict_link.get_no_conflict_link_nodes3(totalnode_comple , start1, end3,time_2_build,N,P)

import draw.basic_functio.conflict_link as conflict_link




In [50]:
import importlib
import draw.basic_functio.motif as motif
import importlib
importlib.reload(motif)
importlib.reload(conflict_link)
importlib.reload(tegnode)

<module 'genaric2.tegnode' from 'C:\\usrspace\\mywork\\generic\\genaric2\\tegnode.py'>

In [72]:
# import draw.basic_functio.transnodes as transnodes
# totalnode_new = transnodes.transnodes_new(totalnode_raw)
ratio=0.3
adjust_flag=1

这里是测试版的，因为调试的时候需要特殊的架构，下面运行后是不会改变totalnode_raw的情况的，因此适合进行调试操作

In [73]:
raw_edges_by_step_onestep,pending_edges_onstep,IG_link_onestep,by_y = conflict_link.get_no_conflict_link_nodes4(totalnode_raw , start_ts_onestep, end_ts_onestep,time_2_build,N,P,ratio,end2,0,adjustflag=adjust_flag)

1
# batch 1 (items 0..0)
# batch 2 (items 1..1)
# batch 3 (items 2..2)
# batch 4 (items 3..3)
# batch 5 (items 4..4)
# batch 6 (items 5..5)


In [42]:
totalnode_raw[0,0,start2]

tegnode(asc_nodes_region_id=False, rightneighbor=(1, 0, 14065), leftneighbor=None, left_state=-1, right_state=1, node_type=1, timelast=0)

In [15]:

import draw.pyqt_draw.pyqt_main2 as pyqt_main2
import importlib
importlib.reload(pyqt_main2)


<module 'draw.pyqt_draw.pyqt_main2' from 'C:\\usrspace\\mywork\\generic\\draw\\pyqt_draw\\pyqt_main2.py'>

In [19]:

# 3) 创建并配置 viewer
viewer =  pyqt_main2.SatelliteViewer(group_data_one_step)
viewer.setWindowTitle("modify one step desgin")
viewer.resize(1200, 700)


viewer.edges_by_step = raw_edges_by_step_onestep
viewer.pending_links_by_step =pending_edges_onstep
viewer.IG_link_by_step =IG_link_onestep

viewer.show()



In [76]:

# 3) 创建并配置 viewer
viewer =  SatelliteViewer(group_data_one_step)
viewer.setWindowTitle("modify one step desgin")
viewer.resize(1200, 700)


viewer.edges_by_step = raw_edges_by_step_onestep
viewer.pending_links_by_step =pending_edges_onstep
viewer.IG_link_by_step =IG_link_onestep

viewer.show()


下面是正式版的，会反向修改totalnode_raw，也就是node1 node2 node3的数值，然后我们就可以将相关的node给存下来了

In [30]:

# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data_one_step)
viewer.setWindowTitle("modify one step desgin")
viewer.resize(1200, 700)


viewer.edges_by_step = raw_edges_by_step_onestep
viewer.pending_links_by_step =pending_edges_onstep
viewer.show()



In [20]:
# 这里显示的是raw，其实就是用来开发时候进行对比的
raw_edges_by_step,raw_pending_edge = motif.transform_nodes_2_rawedge_test(totalnode_raw, P, N, start_ts_onestep, end_ts_onestep)

In [23]:

# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data_one_step)
viewer.setWindowTitle("raw")
viewer.resize(1200, 700)


viewer.edges_by_step = raw_edges_by_step
viewer.pending_links_by_step =raw_pending_edge
viewer.show()
_viewer_list.append(viewer)


In [77]:
modify_raw_edges_by_step_onestep,modify_pending_edges_onstep,IG_link_onestep,by_y = conflict_link.get_no_conflict_link_nodes4(totalnode_raw , start_ts_onestep, end_ts_onestep,time_2_build,N,P,ratio,end2,1,adjustflag=adjust_flag)

1
# batch 1 (items 0..0)
# batch 2 (items 1..1)
# batch 3 (items 2..2)
# batch 4 (items 3..3)
# batch 5 (items 4..4)
# batch 6 (items 5..5)


In [80]:

# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data_one_step)
viewer.setWindowTitle("modify_raw")
viewer.resize(1200, 700)


viewer.edges_by_step = modify_raw_edges_by_step_onestep
viewer.pending_links_by_step =modify_pending_edges_onstep
viewer.IG_link_by_step =IG_link_onestep


viewer.show()




In [79]:
##在检查实际拓扑后，确认无问题后，就将其存入到xml文件里去，注意，我们只要保存异轨链路信息即可，其余不必保存
import draw.basic_functio.write2xml as write2xml
import genaric2.tegnode as tegnode
# 推荐：用 raw string 防止反斜杠转义，并改成有意义的文件名
# 这里，我们要把原始的边转为node进行存储


file_path = MODIFY_DIR / f"interplane_links_{start2}_{end2}.xml"


write2xml.nodes_to_xml_test(
    nodes2,
   file_path
)


In [196]:
#第一个要记得存下来


file_path = MODIFY_DIR / f"interplane_links_{start1}_{end1}.xml"


write2xml.nodes_to_xml_test(
    nodes1,
   file_path
)


In [81]:
start3

20814

In [82]:
end3

22005

In [83]:


file_path = MODIFY_DIR / f"interplane_links_{start3}_{end3}.xml"


write2xml.nodes_to_xml_test(
    nodes3,
   file_path
)
